# 02. Debate 与投票共识

当多个 Agent 对一个问题的看法不一致时，Debate + Voting 是一种有效的聚合机制：

1. 每个 Agent 独立提出观点
2. 多轮交锋：互相 critique
3. Judge（或投票机制）聚合最终结论

> **检查点**：能画出 Multi-Agent Debate 的时序图（Pro → Critique → Counter → Judge）。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-flash  base_url=https://api.deepseek.com


In [2]:
from dataclasses import dataclass

@dataclass
class Verdict:
    winner: str; pro_score: float; con_score: float

class Debate:
    def __init__(self, pro_args: list[str], con_args: list[str]):
        self.pro = pro_args; self.con = con_args
        self.pro_scores = []; self.con_scores = []
    
    def round(self, n: int):
        p = self.pro[(n-1) % len(self.pro)]
        c = self.con[(n-1) % len(self.con)]
        self.pro_scores.append(len(p.split()))  # proxy score: word count
        self.con_scores.append(len(c.split()))
        print(f"R{n} PRO: {p[:60]}...")
        print(f"R{n} CON: {c[:60]}...\n")
    
    def judge(self, strategy: str = "sum") -> Verdict:
        ps = sum(self.pro_scores) if strategy == "sum" else max(self.pro_scores)
        cs = sum(self.con_scores) if strategy == "sum" else max(self.con_scores)
        return Verdict("PRO" if ps>cs else "CON" if cs>ps else "TIE", float(ps), float(cs))

debate = Debate(
    ["Static typing catches bugs at compile time",
     "Better performance with zero-cost abstractions",
     "Cargo is the best package manager"],
    ["Faster development with dynamic typing",
     "Larger ecosystem and more libraries",
     "Lower learning curve for beginners"]
)
for r in range(1, 4):
    debate.round(r)
v = debate.judge()
print(f"Verdict: {v.winner} (PRO={v.pro_score}, CON={v.con_score})")

R1 PRO: Static typing catches bugs at compile time...
R1 CON: Faster development with dynamic typing...

R2 PRO: Better performance with zero-cost abstractions...
R2 CON: Larger ecosystem and more libraries...

R3 PRO: Cargo is the best package manager...
R3 CON: Lower learning curve for beginners...

Verdict: PRO (PRO=18.0, CON=15.0)


## 辩论的陷阱

| 陷阱 | 表现 | 缓解 |
|------|------|------|
| 群体思维 | 所有 Agent 共享同样的错误假设 | 引入 outsider Agent，强制异见 |
| Verbosity Bias | 话多的 Agent 得分高 | 限制发言长度，用内容质量而非字数评分 |
| Position Bias | 先发言的占优势 | 轮换发言顺序 |
| 伪共识 | 表面同意但各有保留 | 要求 Agent 必须指出一个不同点 |

> **检查点**：能命名至少 3 种辩论陷阱及其缓解策略。

## 真实 API 实验

真实辩论：两个 LLM 各持一方，第三个 LLM 当裁判（`debate_demo.py --use-api`）。


In [3]:
SCRIPT = ROOT / 'agents' / 'orchestration' / 'scripts' / 'debate_demo.py'
import subprocess, sys
result = subprocess.run([sys.executable, str(SCRIPT), '--use-api', '--verbose'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)


Debate: 'Which is better for systems programming?' [LLM]

--- Round 1 ---
  [Alpha] PRO: Pro: Rust is better for systems programming because it offers C-level control and perfo...
  [Beta] CON: Con: C remains better for systems programming because it is the established foundation o...

--- Round 2 ---
  [Alpha] PRO: C's incumbency in OSes, firmware, and embedded is real but reflects path dependence, no...
  [Beta] CON: Rust’s compile-time safety is not a universal win for systems programming because the ha...

--- Round 3 ---
  [Alpha] PRO: Rust is better because it doesn’t pretend systems programming can avoid `unsafe`; it is...
  [Beta] CON: Rust’s core advantage shrinks exactly where systems programming is hardest: at unsafe/FF...

Verdict (majority): PRO=6 CON=6 → TIE



## 练习

1. 实现 majority vote、ranked choice、weighted voting 三种聚合策略。在同一组辩论上对比结果。
2. 设计一个「故意引入错误 Agent」的实验：3 个 Agent 中 1 个给出错误信息，观察辩论能否发现。
3. 实现黑板架构：Agent 不直接对话，而是读写共享的「黑板」，如何影响辩论的动态？

> **检查点**：能为你的 Agent 团队设计合理的辩论和投票机制，并避免常见陷阱。